In [5]:
import duckdb
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="ticks")
mm = 1 / 25.4  # centimeters in inches

import sys

sys.path.append("..")

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()
import numpy as np

In [6]:
## Plot for Joint ASAP paper using only cingulate, parietal, and frontal

percentile = 0  # Set the oligomer intensity percentile for the analysis
cells = "microglia"

output_dir = r"/scratch/sycamore-asap/ASAP_Plots"

# Reconnect to the database
conn = duckdb.connect(r"/scratch/duckdb-database/main_survey_database.duckdb")

volume = (np.pi * (4 / 3)) * np.power(5, 3)


# Function to create a boxplot with brain regions on x-axis
def create_region_boxplot(conn, y_metric="incelldens"):
    # Query only data needed for {cells} at specified percentile and only for cingulate, parietal, and frontal regions
    query = f"""
    SELECT 
        brain_region, 
        {y_metric}, 
        disease, 
        donorid
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type = '{cells}' AND percentile = {percentile}
        AND "area/um3" >= 100 AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'frontal', 'caudate')
    """
    subset = conn.execute(query).fetchdf()
    if subset.empty:
        print(f"No data for {cells} at {percentile}th percentile")
        return

    # Rename brain regions
    region_mapping = {
        "caudate": "Caudate",
        "cingulate": "Cingulate",
        "frontal": "Frontal",
    }
    subset["brain_region"] = subset["brain_region"].map(region_mapping)

    subset[y_metric] = subset[y_metric] * volume
    # Define region order (Parietal first, then Cingulate, then Frontal)
    region_order = ["Caudate", "Cingulate", "Frontal"]

    # Calculate donor-level medians for statistical testing
    donor_medians = (
        subset.groupby(["brain_region", "disease", "donorid"])[y_metric]
        .median()
        .reset_index()
    )

    # Build statistics output string
    stats_output = []
    stats_output.append(f"Statistical Testing ({y_metric})")
    stats_output.append("=" * 60)

    # Perform Mann-Whitney U tests for each region (PD vs Control)
    print(f"\nStatistical Testing ({y_metric}):")
    print("=" * 60)
    p_values = {}
    for region in region_order:
        control_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "HC")
        ][y_metric].values
        pd_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "PD")
        ][y_metric].values

        if len(control_data) > 0 and len(pd_data) > 0:
            statistic, p_value = stats.mannwhitneyu(
                control_data, pd_data, alternative="two-sided"
            )
            p_values[region] = p_value

            # Determine significance level
            if p_value < 0.001:
                sig = "***"
            elif p_value < 0.01:
                sig = "**"
            elif p_value < 0.05:
                sig = "*"
            else:
                sig = "ns"

            # Print to console
            print(f"{region}: U={statistic:.2f}, p={p_value:.4f} {sig}")
            print(
                f"  Control: n={len(control_data)}, median={np.median(control_data):.3f}"
            )
            print(f"  PD: n={len(pd_data)}, median={np.median(pd_data):.3f}")

            # Add to stats output
            stats_output.append(f"\n{region}: Mann-Whitney U test (PD vs Control)")
            stats_output.append(f"  U statistic = {statistic:.2f}")
            stats_output.append(f"  p-value = {p_value:.4f} {sig}")
            stats_output.append(
                f"  Control: n={len(control_data)}, median={np.median(control_data):.3f}, mean={np.mean(control_data):.3f}, SD={np.std(control_data, ddof=1):.3f}"
            )
            stats_output.append(
                f"  PD: n={len(pd_data)}, median={np.median(pd_data):.3f}, mean={np.mean(pd_data):.3f}, SD={np.std(pd_data, ddof=1):.3f}"
            )
        else:
            p_values[region] = None
            print(f"{region}: Insufficient data")
            stats_output.append(f"\n{region}: Insufficient data")
    print("=" * 60)
    stats_output.append("\n" + "=" * 60)

    # Save statistics to text file
    stats_filename = f"{output_dir}/{cells}_{percentile}percentile_{y_metric}_by_region_3D_boxplot_stats.txt"
    with open(stats_filename, "w") as f:
        f.write("\n".join(stats_output))
    print(f"Saved statistics to {stats_filename}")

    # Set seaborn style
    sns.set_theme(style="ticks")

    # Create figure
    fig, ax = plotter.two_column_plot(height=(170 / 4) * mm, width=180 * mm, lw=0.75)

    # Create custom palette and rename HC to Control
    palette = {"PD": "#E88247", "Control": "#C3B3A1"}
    hue_order = ["Control", "PD"]  # Define explicit order

    # Replace 'HC' with 'Control' in the dataset
    subset["disease"] = subset["disease"].replace("HC", "Control")

    # Create boxplot with custom region order
    ax = sns.boxplot(
        data=subset,
        x="brain_region",
        y=y_metric,
        hue="disease",
        hue_order=hue_order,
        palette=palette,
        showfliers=False,  # Don't show outliers
        ax=ax,
        order=region_order,
    )

    # Now add scatter points for individual donors with jittering on top
    for i, region in enumerate(region_order):
        region_data = subset[subset["brain_region"] == region]

        # For each disease
        for j, disease in enumerate(hue_order):
            disease_data = region_data[region_data["disease"] == disease]
            donor_medians_plot = (
                disease_data.groupby("donorid")[y_metric].median().reset_index()
            )

            if donor_medians_plot.empty:
                continue

            # Calculate position for this box
            num_hues = len(hue_order)
            offset = 0.4  # Total width of hue groups
            pos_offset = (j - (num_hues - 1) / 2) * (offset / num_hues)
            x_pos_base = i + pos_offset

            for k, median_val in enumerate(donor_medians_plot[y_metric]):
                # Add random jitter
                jitter = np.random.uniform(-0.05, 0.05)

                # Scatter plot for donor medians
                plt.scatter(
                    x_pos_base + jitter,
                    median_val,
                    color=palette[disease],
                    edgecolor="black",
                    s=30,
                    alpha=0.9,
                    zorder=10,  # Ensure points are on top
                )

    # Add significance stars or ns
    y_max = subset[y_metric].max()
    y_range = subset[y_metric].max() - subset[y_metric].min()
    star_height = y_max + y_range * 0.05

    for i, region in enumerate(region_order):
        if region in p_values and p_values[region] is not None:
            p_val = p_values[region]
            if p_val < 0.001:
                sig_text = "***"
                text_fontsize = 8
            elif p_val < 0.01:
                sig_text = "**"
                text_fontsize = 8
            elif p_val < 0.05:
                sig_text = "*"
                text_fontsize = 8
            else:
                sig_text = "ns"
                text_fontsize = 6

            # Draw horizontal line connecting the two groups
            x1, x2 = i - 0.2, i + 0.2
            ax.plot([x1, x2], [star_height, star_height], "k-", lw=1)
            # Add stars or ns
            ax.text(
                i,
                star_height + y_range * 0.02,
                sig_text,
                ha="center",
                va="bottom",
                fontsize=text_fontsize,
                fontweight="bold",
            )

    ax.tick_params(labelsize=7)
    # Calculate stats for each region and disease combination
    legend_elements = []

    # Remove default legend
    ax.get_legend().remove()

    # Create custom legend with donor and cell counts for each region and disease
    for region in region_order:
        for disease in hue_order:
            region_disease_data = subset[
                (subset["brain_region"] == region) & (subset["disease"] == disease)
            ]
            if not region_disease_data.empty:
                unique_donors = region_disease_data["donorid"].nunique()
                cell_count = len(region_disease_data)
                region_short = (
                    "PARI"
                    if region == "Parietal"
                    else (
                        "CIN"
                        if region == "Cingulate"
                        else (
                            "FRO"
                            if region == "Frontal"
                            else (
                                "PARA"
                                if region == "Parahippocampus"
                                else "TEMP" if region == "Temporal" else "CAUD"
                            )
                        )
                    )
                )
                label = f"{disease} {region_short} ($\\it{{N}}$ = {unique_donors}; $\\it{{n}}_{{cells}}$ = {cell_count:,})"
                legend_elements.append(
                    Patch(facecolor=palette[disease], edgecolor="black", label=label)
                )

    # Add the custom legend
    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=6,
        ncols=1,
        bbox_to_anchor=(1.32, 1.05),
    )

    # Set labels and title
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.xticks(rotation=0, ha="right")

    # Set y-limit for puncta_cell_likelihood
    if y_metric == "puncta_cell_likelihood":
        plt.ylim(0, 10)

    # Set appropriate y-label
    if y_metric == "incelldens":
        plt.ylabel(r"αSyn oligomers per microglia", fontsize=8)
    elif y_metric == "puncta_cell_likelihood":
        plt.ylabel(f"Relative In-cell Density (spots/µm$^3$)", fontsize=8)
    else:
        plt.ylabel(y_metric, fontsize=8)
    ax.set(xlabel=None)
    # Save the plot
    filename = f"{output_dir}/{cells}_{percentile}percentile_{y_metric}_by_region_3D_boxplot.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()

    # Free memory
    del subset
    print(f"Saved plot to {filename}")


# Create plots for both metrics
create_region_boxplot(conn, "incelldens")

# Close the connection
conn.close()


Statistical Testing (incelldens):
Caudate: U=89.00, p=0.3247 ns
  Control: n=11, median=60.724
  PD: n=13, median=65.267
Cingulate: U=33.00, p=0.0053 **
  Control: n=13, median=20.051
  PD: n=14, median=65.506
Frontal: U=92.00, p=0.9806 ns
  Control: n=13, median=127.719
  PD: n=14, median=118.511
Saved statistics to /scratch/sycamore-asap/ASAP_Plots/microglia_0percentile_incelldens_by_region_3D_boxplot_stats.txt
Saved plot to /scratch/sycamore-asap/ASAP_Plots/microglia_0percentile_incelldens_by_region_3D_boxplot.svg


In [8]:
# Constants
percentile = 0  # Set the oligomer intensity percentile for the analysis
cells = "microglia"
volume = (np.pi * (4 / 3)) * np.power(5, 3)

# Reconnect to the database
conn = duckdb.connect(r"/scratch/duckdb-database/main_survey_database.duckdb")

from scipy.stats import sem


def calculate_ratio_plot(conn, y_metric="incelldens"):
    # Query data needed for {cells} at specified percentile and only for selected regions
    query = f"""
    SELECT 
        brain_region, 
        {y_metric}, 
        disease, 
        donorid
    FROM 
        PCL_dens_percell_3D
    WHERE 
        cell_type = '{cells}' AND percentile = {percentile}
        AND "area/um3" >= 100 AND "area/um3" <= 500
        AND brain_region IN ('cingulate', 'frontal', 'caudate')
    """
    subset = conn.execute(query).fetchdf()
    if subset.empty:
        print(f"No data for {cells} at {percentile}th percentile")
        return

    # Rename brain regions and diseases
    region_mapping = {
        "caudate": "Caudate",
        "cingulate": "Cingulate",
        "frontal": "Frontal",
    }
    subset["brain_region"] = subset["brain_region"].map(region_mapping)
    subset["disease"] = subset["disease"].replace("HC", "Control")

    # Scale the y_metric by volume
    subset[y_metric] = subset[y_metric] * volume

    # Define region order
    region_order = ["Caudate", "Cingulate", "Frontal"]

    # Calculate median values for each donor in each region and disease
    donor_medians = (
        subset.groupby(["brain_region", "disease", "donorid"])[y_metric]
        .median()
        .reset_index()
    )

    # Calculate ratio of PD to Control medians for each region
    ratios = []
    ratio_errors = []
    n_donors = []
    all_region_ratios = {}  # Store individual ratios for statistical testing

    for region in region_order:
        # Get control medians
        control_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "Control")
        ]
        control_vals = control_data[y_metric].values

        # Get PD medians
        pd_data = donor_medians[
            (donor_medians["brain_region"] == region)
            & (donor_medians["disease"] == "PD")
        ]
        pd_vals = pd_data[y_metric].values

        if len(control_vals) == 0 or len(pd_vals) == 0:
            ratios.append(np.nan)
            ratio_errors.append(np.nan)
            n_donors.append(0)
            all_region_ratios[region] = np.array([])
            continue

        # Calculate ratio for each PD donor relative to control median
        control_median = np.median(control_vals)
        individual_ratios = pd_vals / control_median

        # Store the mean ratio and SD across PD donors
        ratios.append(np.mean(individual_ratios))
        ratio_errors.append(
            np.std(individual_ratios, ddof=1)
        )  # Use standard deviation instead of SEM
        n_donors.append(len(pd_vals))
        all_region_ratios[region] = individual_ratios

    # Build statistics output string
    stats_output = []
    stats_output.append(f"Statistical Testing for Ratios ({y_metric})")
    stats_output.append("=" * 60)

    # Perform Kruskal-Wallis test to compare ratios between regions
    print(f"\nStatistical Testing for Ratios ({y_metric}):")
    print("=" * 60)

    # Print summary statistics for each region
    print("\nSummary Statistics by Region:")
    stats_output.append("\nSummary Statistics by Region:")

    for region in region_order:
        if len(all_region_ratios[region]) > 0:
            region_ratios = all_region_ratios[region]

            # Print to console
            print(f"{region}:")
            print(f"  n = {len(region_ratios)}")
            print(f"  Mean ratio = {np.mean(region_ratios):.3f}")
            print(f"  Median ratio = {np.median(region_ratios):.3f}")
            print(f"  SD = {np.std(region_ratios, ddof=1):.3f}")
            print(f"  SEM = {sem(region_ratios):.3f}")
            print(
                f"  Range = [{np.min(region_ratios):.3f}, {np.max(region_ratios):.3f}]"
            )

            # Add to stats output
            stats_output.append(f"\n{region}:")
            stats_output.append(f"  n = {len(region_ratios)}")
            stats_output.append(f"  Mean ratio = {np.mean(region_ratios):.3f}")
            stats_output.append(f"  Median ratio = {np.median(region_ratios):.3f}")
            stats_output.append(f"  SD = {np.std(region_ratios, ddof=1):.3f}")
            stats_output.append(f"  SEM = {sem(region_ratios):.3f}")
            stats_output.append(
                f"  Range = [{np.min(region_ratios):.3f}, {np.max(region_ratios):.3f}]"
            )

    # Filter out regions with no data
    valid_ratios = [
        all_region_ratios[region]
        for region in region_order
        if len(all_region_ratios[region]) > 0
    ]
    valid_regions = [
        region for region in region_order if len(all_region_ratios[region]) > 0
    ]

    pairwise_results = []
    if len(valid_ratios) >= 2:
        h_stat, kw_p_value = stats.kruskal(*valid_ratios)

        # Print to console
        print(f"\nKruskal-Wallis H-test (comparing all regions):")
        print(f"  H = {h_stat:.3f}")
        print(f"  p-value = {kw_p_value:.4f}")

        # Add to stats output
        stats_output.append(f"\nKruskal-Wallis H-test (comparing all regions):")
        stats_output.append(f"  H = {h_stat:.3f}")
        stats_output.append(f"  p-value = {kw_p_value:.4f}")

        if kw_p_value < 0.05:
            print(f"  Result: Significant difference between regions (p < 0.05)")
            stats_output.append(
                f"  Result: Significant difference between regions (p < 0.05)"
            )
        else:
            print(f"  Result: No significant difference between regions (p >= 0.05)")
            stats_output.append(
                f"  Result: No significant difference between regions (p >= 0.05)"
            )

        print("\nPairwise comparisons (Mann-Whitney U with Bonferroni correction):")
        stats_output.append(
            "\nPairwise comparisons (Mann-Whitney U with Bonferroni correction):"
        )

        # Perform pairwise comparisons
        from itertools import combinations

        n_comparisons = len(list(combinations(valid_regions, 2)))
        bonferroni_alpha = 0.05 / n_comparisons

        print(f"  Number of comparisons: {n_comparisons}")
        print(f"  Bonferroni-corrected α: {bonferroni_alpha:.4f}")
        print()

        stats_output.append(f"  Number of comparisons: {n_comparisons}")
        stats_output.append(f"  Bonferroni-corrected α: {bonferroni_alpha:.4f}")
        stats_output.append("")

        for r1, r2 in combinations(valid_regions, 2):
            stat, p_val = stats.mannwhitneyu(
                all_region_ratios[r1], all_region_ratios[r2], alternative="two-sided"
            )
            is_sig = p_val < bonferroni_alpha
            if p_val < 0.001:
                sig_text = "***"
            elif p_val < 0.01:
                sig_text = "**"
            elif p_val < 0.05:
                sig_text = "*"
            else:
                sig_text = "ns"

            pairwise_results.append((r1, r2, p_val, is_sig, sig_text))

            # Print to console
            print(f"  {r1} vs {r2}:")
            print(f"    U = {stat:.2f}")
            print(f"    p-value = {p_val:.4f} ({sig_text})")
            print(f"    Significant (Bonferroni): {'Yes' if is_sig else 'No'}")
            print()

            # Add to stats output
            stats_output.append(f"  {r1} vs {r2}:")
            stats_output.append(f"    U = {stat:.2f}")
            stats_output.append(f"    p-value = {p_val:.4f} ({sig_text})")
            stats_output.append(
                f"    Significant (Bonferroni): {'Yes' if is_sig else 'No'}"
            )
            stats_output.append("")
    else:
        print("Not enough regions with data for statistical comparison")
        stats_output.append("Not enough regions with data for statistical comparison")
        kw_p_value = None

    print("=" * 60)
    stats_output.append("=" * 60)

    # Save statistics to text file
    stats_filename = (
        f"{output_dir}/{cells}_{percentile}percentile_{y_metric}_ratio_plot_stats.txt"
    )
    with open(stats_filename, "w") as f:
        f.write("\n".join(stats_output))
    print(f"Saved statistics to {stats_filename}")

    # Create the plot
    sns.set_theme(style="ticks")
    fig, ax = plotter.two_column_plot(
        height=(170 / 4) * mm, width=(180 / 3.6) * mm, lw=0.75
    )

    # Plot the ratios with error bars
    x_pos = np.arange(len(region_order))
    ax.bar(
        x_pos,
        ratios,
        yerr=ratio_errors,
        color="#E88247",
        edgecolor="black",
        width=0.6,
        capsize=5,
        error_kw={"elinewidth": 1, "capthick": 1},
    )

    # Add reference line at 1.0 (no difference)
    ax.axhline(y=1.0, color="#C3B3A1", linestyle="--", linewidth=1)

    # Add pairwise comparison brackets and stars
    if len(pairwise_results) > 0:
        y_max = max([r + e for r, e in zip(ratios, ratio_errors) if not np.isnan(r)])

        # Create a mapping of region names to x positions
        region_to_x = {region: i for i, region in enumerate(region_order)}

        # Draw brackets for significant pairwise comparisons
        bracket_height_start = y_max * 1.15
        bracket_height_increment = y_max * 0.15

        sig_comparisons = [
            (r1, r2, sig) for r1, r2, p, is_sig, sig in pairwise_results if is_sig
        ]

        for idx, (r1, r2, sig_text) in enumerate(sig_comparisons):
            x1 = region_to_x[r1]
            x2 = region_to_x[r2]

            # Calculate bracket height (stack them if multiple)
            bracket_y = bracket_height_start + idx * bracket_height_increment

            # Draw the bracket
            ax.plot(
                [x1, x1, x2, x2],
                [
                    bracket_y - 0.02 * y_max,
                    bracket_y,
                    bracket_y,
                    bracket_y - 0.02 * y_max,
                ],
                "k-",
                lw=1,
            )

            # Add significance stars
            ax.text(
                (x1 + x2) / 2,
                bracket_y + 0.02 * y_max,
                sig_text,
                ha="center",
                va="bottom",
                fontsize=7,
                fontweight="bold",
            )

        # Add Kruskal-Wallis p-value if significant
        if kw_p_value is not None and kw_p_value < 0.05:
            max_bracket_y = (
                bracket_height_start + len(sig_comparisons) * bracket_height_increment
            )
            ax.text(
                len(region_order) / 2 - 0.5,
                max_bracket_y + 0.1 * y_max,
                f"KW: p={kw_p_value:.3f}",
                ha="center",
                va="bottom",
                fontsize=6,
                style="italic",
            )

    # Set plot details
    ax.set_xticks(x_pos)
    ax.set_xticklabels(region_order)
    ax.tick_params(labelsize=7)

    # Add sample size information
    for i, n in enumerate(n_donors):
        ax.text(x_pos[i], 0.1, f"n={n}", ha="center", va="bottom", fontsize=6)

    # Set labels and title
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.25)
    plt.ylabel("PD/Control ratio", fontsize=8)
    ax.set(xlabel=None)

    # Set appropriate title based on y_metric
    if y_metric == "incelldens":
        plt.title("Ratio of αSyn oligomers per microglia", fontsize=9)
    elif y_metric == "puncta_cell_likelihood":
        plt.title("Ratio of relative in-cell density", fontsize=9)

    # Save the plot
    filename = f"{output_dir}/{cells}_{percentile}percentile_{y_metric}_ratio_plot.svg"
    plt.savefig(filename, dpi=1200, format="svg", bbox_inches="tight")
    plt.close()

    print(f"Saved ratio plot to {filename}")


# Create ratio plots for both metrics
calculate_ratio_plot(conn, "incelldens")

# Close the connection
conn.close()


Statistical Testing for Ratios (incelldens):

Summary Statistics by Region:
Caudate:
  n = 13
  Mean ratio = 1.116
  Median ratio = 1.075
  SD = 0.649
  SEM = 0.180
  Range = [0.242, 2.104]
Cingulate:
  n = 14
  Mean ratio = 3.671
  Median ratio = 3.267
  SD = 2.480
  SEM = 0.663
  Range = [0.344, 9.774]
Frontal:
  n = 14
  Mean ratio = 0.990
  Median ratio = 0.928
  SD = 0.476
  SEM = 0.127
  Range = [0.363, 2.032]

Kruskal-Wallis H-test (comparing all regions):
  H = 15.627
  p-value = 0.0004
  Result: Significant difference between regions (p < 0.05)

Pairwise comparisons (Mann-Whitney U with Bonferroni correction):
  Number of comparisons: 3
  Bonferroni-corrected α: 0.0167

  Caudate vs Cingulate:
    U = 26.00
    p-value = 0.0017 (**)
    Significant (Bonferroni): Yes

  Caudate vs Frontal:
    U = 100.00
    p-value = 0.6800 (ns)
    Significant (Bonferroni): No

  Cingulate vs Frontal:
    U = 176.00
    p-value = 0.0004 (***)
    Significant (Bonferroni): Yes

Saved statisti